# Robots Vote: Do Robots Influence Your Vote?
**Experiment support code** <br>
This code supports the experiment, allowing us to control the behavior of the robots. 

## Getting Started...
Imports and auxiliary code

In [1]:
import asyncio
import soundfile as sf
import numpy as np
from reachy_mini import ReachyMini
from dash.robot import DashRobot, discover_and_connect
import time
import random

class Robot:
    def __init__(self, name, typer):
        self.name = name
        self.type = typer
        self.device = None
        self.audio_path=None
        self.audio_slot=None
        self.state = "idle"

    async def connect(self):
        try:
            if self.type == "dash": 
                
                robo = await discover_and_connect([self.name])
                if robo:
                    self.device = robo
            
                
            elif self.type == "reachy":
                print(f"[{self.name}] A procurar Reachy...")
                #self.device = ReachyMini(id=self.name)
                if name == "Robot1":
                    self.device = ReachyMini(host="reachy-mini-one.local")
                elif name == "Robot2":
                    self.device = ReachyMini(host="reachy-mini-two.local")
                else:
                    self.device = ReachyMini()
                self.device.media.start_playing()
            else:
                print("RobotSim " + self.name + " is connected!")
            print(f"✅ {self.name} connected")
        except Exception as e:
                print(f"❌ Robot {self.name} falied to connect: {e}")

            

    def is_connected(self):
        return self.device is None or (self.type != "dash" and self.type != "reachy")
    
    def change_audio(self, audio):
        if self.type == "dash":
            self.audio_slot=audio
        elif self.type == "reachy":
            self.audio_path=audio

    async def animete_and_talk(self,duration):
        if self.type == "dash": 
            await self.device.say(self.audio_slot) 
            end = time.time() + duration
            while time.time() < end:
                for brilho in [0, 50, 150, 255, 150, 50]:
                        if not robot.client.is_connected: break
                        await robot.eye_brightness(brilho)
                        await asyncio.sleep(0.05)
        if self.type == "reachy":
            data, samplerate = sf.read(file_path)

            if len(data.shape) > 1:
                data = data.mean(axis=1)
        
            self.device.media.start_playing()
        
            samples_per_step = int(samplerate * STEP_TIME)
        
            for i in range(0, len(data), samples_per_step):
                start_time = time.monotonic()
        
                chunk = data[i:i + samples_per_step]
        
                # 🎧 enviar áudio
                self.device.media.push_audio_sample(chunk)
        
                if len(chunk) > 0:
                    rms = np.sqrt(np.mean(chunk**2))
        
                    pitch = (rms ** 0.5) * SENSITIVITY * 50
                    print(pitch)
                    roll = np.random.uniform(-50, 50) * rms
                    yaw = np.random.uniform(-50, 50) * rms
        
                    self.device.goto_target(
                        head=create_head_pose(
                            roll=roll,
                            pitch=pitch,
                            yaw=yaw,
                            degrees=True
                        ),
                        duration=STEP_TIME  # igual ao intervalo!
                    )
        
                # ⏱️ sync
                elapsed = time.monotonic() - start_time
                sleep_time = STEP_TIME - elapsed
        
                if sleep_time > 0:
                    asyncio.sleep(sleep_time)
        
            # reset
            self.device.goto_target(head=create_head_pose(), duration=1.0)

        else: 
            end = time.time() + duration
            while time.time() < end:
                print("RobotSim " + self.name + " is talking!")
                await asyncio.sleep(1.5)
                


    async def idle_animation(self, duration):
        end = time.time() + duration
        yaw = 0
        while time.time() < end:
            if self.type == "dash": 
                
                yaw = min(14,max(-14,yaw+random.randint(-2, 2)))
                await self.device.head_yaw(yaw)
                await asyncio.sleep(random.uniform(2.5, 4.0))
            
            elif self.type == "reachy":    
                pitch = random.uniform(-8, 8)
                yaw = random.uniform(-8, 8)
                roll = random.uniform(-8, 8)
        
                duration_anim = random.uniform(1.5, 3.0)
        
                self.device.goto_target(
                    head=create_head_pose(pitch=pitch, roll=roll, yaw=yaw, degrees=True),
                    duration=duration_anim
                )
        
                await asyncio.sleep(duration_anim + random.uniform(0.5, 1.5))

            else: 
                print("RobotSim " + self.name + " is idling!")
                await asyncio.sleep(random.uniform(2.5, 4.0))

            

    async def listening_animation(self, duration):
        end = time.time() + duration
        
        while time.time() < end:
            
            if self.type == "reachy":  
                for brilho in ["#000000", "#ffffff", "#000000", "#ffffff", "#000000", "#ffffff", "#000000", "#ffffff",]:
                    await self.device.left_ear_color(brilho)
                    await self.device.right_ear_color(brilho)
                    await asyncio.sleep(1.5)
                
            elif self.type == "dash": 
                self.device.goto_target(
                    antennas=[np.deg2rad(-30), np.deg2rad(-15)],
                    duration=1.0,
                )
            
                await asyncio.sleep(1.5)
            
                mini.goto_target(
                    antennas=[np.deg2rad(15), np.deg2rad(30)],
                    duration=1.0,
                )
            
                await asyncio.sleep(1.5)

            else:
                print("RobotSim " + self.name + " is listening!")
                await asyncio.sleep(1.5)
        

    async def disconnect(self):
        if self.type == "dash": 
            await self.device.disconnect()
            print("Dash " + self.name + " disconnected!")
        elif self.type == "reachy":
            self.device = None
            print("Reachy " + self.name + " disconnected!")
        else:
            print("RobotSim " + self.name + " disconnected!")

    def is_speaking(self):
        return self.state == "speak"
    
    def set_state(self, state):
        self.state = state
        

    async def run_loop(self):
        print(self.state)
        while True:
            match self.state:
                case "idle":
                    task1 = asyncio.create_task(self.idle_animation(10))
                    while (not task1.done()):
                        await asyncio.sleep(1)
                case "listen":
                    print("Listening")
                    task1 = asyncio.create_task(self.idle_animation(10))
                    task2 = asyncio.create_task(self.listening_animation(10))
                    while (not task1.done() and not task2.done()):
                        await asyncio.sleep(1)
                case "speak":
                    task2 = asyncio.create_task(self.animete_and_talk(15))
                    while (not task2.done()):
                        await asyncio.sleep(1)
                    self.state = "listen"
                    
                case "stop":
                    self.state = "idle"
                    break
                case _:
                    return "Something's wrong with "+self.name
        


In [2]:
reachy = Robot("robot1","")
reachy.change_audio("test1.wav")
await reachy.connect()
asyncio.create_task(reachy.run_loop())
await asyncio.sleep(10)
reachy.set_state("listen")
await asyncio.sleep(10)
reachy.set_state("stop")


RobotSim robot1 is connected!
✅ robot1 connected
idle
RobotSim robot1 is idling!
RobotSim robot1 is idling!
RobotSim robot1 is idling!
RobotSim robot1 is idling!
Listening
RobotSim robot1 is idling!
RobotSim robot1 is listening!
RobotSim robot1 is listening!
RobotSim robot1 is listening!
RobotSim robot1 is idling!
RobotSim robot1 is listening!
RobotSim robot1 is idling!
RobotSim robot1 is listening!
RobotSim robot1 is listening!
RobotSim robot1 is idling!
RobotSim robot1 is listening!


In [2]:
import speech_recognition as sr
import whisper
import torch
from sentence_transformers import SentenceTransformer, util
import time
import os
import numpy as np


def processar_com_whisper(statement_id, statements, robots):
    rec = sr.Recognizer()
    
    rec.pause_threshold = 2.0  # Espera 3s de silêncio antes de processar
    rec.dynamic_energy_threshold = True 
    
    mic = sr.Microphone()

    with mic as source:
        print("\n[CALIBRAÇÃO] Silêncio na sala...")
        rec.adjust_for_ambient_noise(source, duration=2)
        print("SISTEMA ATIVO (Whisper Local). O participante pode entrar na sala...")
        
        while True:
            try:
                # Ouve o áudio e guarda em memória
                print("\nÀ escuta... O statement pode ser lido a qualquer momento...")
                audio = rec.listen(source, timeout=None, phrase_time_limit=15)
                
                print("-> Áudio capturado. Whisper a processar...")
                
                for robot in robots.values():
                    robot.set_state("listen")

                audio_data = audio.get_raw_data(convert_rate=16000, convert_width=2)
                audio_np = np.frombuffer(audio_data, dtype=np.int16).astype(np.float32) / 32768.0
                
                # Transcreve diretamente do array (o Whisper suporta isto)
                result = model_stt.transcribe(audio_np, language="pt")
            
                texto = result["text"].lower().strip()
                
                if len(texto.split()) < 3:
                    continue

                print(f"O Whisper ouviu: '{texto}'")

                # Comparação Semântica
                emb_ouvido = model_sim.encode(texto, convert_to_tensor=True)
                cos_sim = util.cos_sim(emb_ouvido, embeddings_ref)[0]
                idx = cos_sim.argmax().item()
                conf = cos_sim[idx].item()

                print(statement_id)
                print(f"Confiança Semântica: {conf:.2f}")
                if conf > 0.65 and (idx + 1) == int(statement_id):
                    print(f"✅ SUCESSO: Statement {idx + 1}: {statements['robot_related'][idx]}")                   
                    break


            except Exception as e:
                print(f"Erro: {e}")


## Experiment

### Step 1 - Randomization of the experiment
Run this code to randomize the experience

In [3]:
import random
import json

statements = {
    "economic":[],
    "social":[],
    "robot_related":[
    "Os robôs são necessários porque podem fazer trabalhos que são demasiado difíceis ou perigosos para as pessoas.",
    "Os robôs podem tornar a vida mais fácil.",
    "Atribuir tarefas rotineiras aos robôs permite que as pessoas realizem tarefas mais significativas.",
    "As tarefas perigosas devem ser atribuídas prioritariamente aos robôs.",
    "O uso generalizado de robôs vai tirar o emprego às pessoas.",
    "O uso não regulamentado da robótica pode levar a convulsões sociais.",
    "Acabou esta discussão, obrigado a todos."]
}

#topics = ["economic", "social", "robot_related"]
topics = [ "robot_related"]
with open('matrix.json') as f:
    json_data = json.load(f)

with open('participant.json') as f:
    participant_data = json.load(f)


topic = random.choice(topics)
    
print("Tópico: "+ topic)

        
dic_aux = {
    "robots": ["all_disagree", "half_disagree"],
    "virtuals": ["all_disagree", "half_disagree"],
    "articles": ["all_disagree", "half_disagree"]
}
    
ids = [1, 2, 3, 4, 5, 6]
random.shuffle(ids)  # mistura os IDs
    
resultado = {}
i = 0
    
for key, values in dic_aux.items():
    resultado[key] = {}
    for v in values:
        resultado[key][v] = ids[i]
        i += 1
            
conditions = []
    
for categoria, condicoes in resultado.items():
    for condicao, statement_id in condicoes.items():
            
        conditions.append([str(statement_id),categoria,condicao])

random.shuffle(conditions)
print("\nOs statements vão ser discutidos na seguinte ordem: ")
for statement_id,agents,condition in conditions:
    print(f"O statement {statement_id} vai ser discutido com {agents} na condição {condition}.")
    


Tópico: robot_related

Os statements vão ser discutidos na seguinte ordem: 
O statement 2 vai ser discutido com robots na condição all_disagree.
O statement 5 vai ser discutido com robots na condição half_disagree.
O statement 4 vai ser discutido com virtuals na condição half_disagree.
O statement 1 vai ser discutido com virtuals na condição all_disagree.
O statement 6 vai ser discutido com articles na condição half_disagree.
O statement 3 vai ser discutido com articles na condição all_disagree.


## 2 - Experiment Rounds

In [ ]:

i = 1
for statement_id,agents,condition in conditions:
    print(f"{i} - O statement {statement_id} vai ser discutido com {agents} na condição {condition}.")
    i+=1
r = input("\nEscolhe qual das rondas queres fazer:")


print("\nVamos começar...")
statement_id,agents,condition = conditions[int(r)-1]


print("\nStatement: " + statement_id)

        
if agents == "robots":
    print("A carregar Whisper (base)...")
    model_stt = whisper.load_model("small") # Modelo de voz (corre no teu PC)
    print("A carregar Sentence-BERT...")
    model_sim = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    
    embeddings_ref = model_sim.encode(statements[topic], convert_to_tensor=True)
    print("\nColocar os robos seguintes na mesa: ")
    robots = {}
    for robot in json_data[topic][str(agents)][statement_id]["participant_disagree"][condition].keys():
        print(robot)
        typer = json_data[topic][str(agents)][statement_id]["participant_disagree"][condition][robot]["type"]
        #print(typer)
        robots[robot] = Robot(robot,"")
                

    input("\nCarrega Enter para continuar, quando os robos estiverem a postos e ligados...")

    print("\nA conectar os robos...")

    while True:
        for robot in robots.values():
            if not robot.is_connected():
                robot.connect()
        all_connected = all(robot.is_connected() for robot in robots.values())
        if all_connected:
            break

    print("\nRobos connectados")
    print("\nA iniciar os robos...")
    for robot in robots.values():
        asyncio.create_task(robot.run_loop())

    print("\nRobos iniciados... O participante pode entrar na sala.")
    
    task = asyncio.create_task(asyncio.to_thread(processar_com_whisper, statement_id, statements, robots))
    while not task.done():
        await asyncio.sleep(1)
    

    #Robos a responder
    for robot in robots.values():
        robot.set_state("speak")
        while robot.is_speaking():
            await asyncio.sleep(1)

    print("\nVEz do participante... a esperar que o investigador acabe")
                

            
    task = asyncio.create_task(asyncio.to_thread(processar_com_whisper, statement_id, statements, robots))
    while not task.done():
        await asyncio.sleep(1)

    for robot in robots.values():
        robot.set_state("idle")

    input("\nCarrega Enter para finalizar...")

            

            

            

            
                

            
            
            

    

1 - O statement 2 vai ser discutido com robots na condição all_disagree.
2 - O statement 5 vai ser discutido com robots na condição half_disagree.
3 - O statement 4 vai ser discutido com virtuals na condição half_disagree.
4 - O statement 1 vai ser discutido com virtuals na condição all_disagree.
5 - O statement 6 vai ser discutido com articles na condição half_disagree.
6 - O statement 3 vai ser discutido com articles na condição all_disagree.



Escolhe qual das rondas queres fazer: 2



Vamos começar...

Statement: 5
A carregar Whisper (base)...
A carregar Sentence-BERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Colocar os robos seguintes na mesa: 
Robot1
Robot7
Robot2
Robot8



Carrega Enter para continuar, quando os robos estiverem a postos e ligados... 



A conectar os robos...

Robos connectados

A iniciar os robos...

Robos iniciados... O participante pode entrar na sala.
idle
idle
idle
idle
RobotSim Robot1 is idling!
RobotSim Robot7 is idling!
RobotSim Robot2 is idling!
RobotSim Robot8 is idling!

[CALIBRAÇÃO] Silêncio na sala...
SISTEMA ATIVO (Whisper Local). O participante pode entrar na sala...

À escuta... O statement pode ser lido a qualquer momento...
RobotSim Robot8 is idling!
RobotSim Robot1 is idling!
RobotSim Robot2 is idling!
RobotSim Robot7 is idling!
RobotSim Robot7 is idling!
RobotSim Robot2 is idling!
RobotSim Robot1 is idling!
RobotSim Robot8 is idling!
RobotSim Robot2 is idling!
RobotSim Robot7 is idling!
RobotSim Robot8 is idling!
RobotSim Robot1 is idling!
RobotSim Robot2 is idling!


## Economic

### Question 1 - ""

#### Total opposition

In [ ]:

for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my1") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my1") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

### Question 2 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my2") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my2") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

### Question 3 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my3") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my3") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

## Social

### Question 1 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my4") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my4") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

### Question 2 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my5") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my5") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

### Question 3 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my6") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my6") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

## Robot-Related

### Question 1 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my7") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my7") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

### Question 2 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my8") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my8") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

### Question 3 - ""

#### Total opposition

In [ ]:
for name in total_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my9") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

#### Partial opposition

In [ ]:
for name in partial_opposition_robots:
    if name in robots:
        print(f"Robô {nome} a falar...")
        
        await robos[nome].say("my9") 
        
        await animar_olho(robos[nome], duracao=10.0)
        
        print(f"Robô {nome} terminou.")

print("Sequência finalizada!")

## Partial opposition

### Economic

In [ ]:
if "Fernanda" in robos:
    # 1. Inicia o som
    await robos["Fernanda"].say("my2") 
    # 2. Roda a animação sem travar o código (create_task)
    task = asyncio.create_task(animar_olho(robos["Fernanda"], duracao=10.0))
    # 3. Espera a animação terminar antes de você ir para a próxima célula
    await task

### Social

### Robot-Related

### Participant talking

In [ ]:
if "Fernanda" in robos:
    # 1. Inicia o som
    await robos["Fernanda"].say("my2") 
    # 2. Roda a animação sem travar o código (create_task)
    task = asyncio.create_task(mode_listening(robos["Fernanda"], duracao=10.0))
    # 3. Espera a animação terminar antes de você ir para a próxima célula
    await task

## Total opposition


### Economic

### Social

### Robot-Related

### Participant talking

In [1]:
import asyncio
from dash.robot import DashRobot, discover_and_connect
import time

async def main():
    robot = await discover_and_connect(["Fernanda"])
    print(robot)
    if robot:
        print(f"Connected to {robot.address}")
        # Example command to move Dash
        await robot.say("my1")  # Move forward 100mm at 100mm/s
        

if __name__ == "__main__":
    await main()

Connected to CD:2C:91:2D:0D:0C
Connected to CD:2C:91:2D:0D:0C


In [6]:
robot = await discover_and_connect(["Fernanda"])
await robot.disconnect()

CancelledError: 

In [3]:
import asyncio
from dash.robot import DashRobot, discover_and_connect
import time

# Dicionário global para manter os robôs conectados entre as células
robos = {}

async def conectar_equipe():
    nomes = ["Fernanda"] # Substitua pelos nomes reais
    print(f"Buscando por: {', '.join(nomes)}...")
    
    for nome in nomes:
        try:
            # Tenta conectar a cada um individualmente para melhor controle
            robo = await discover_and_connect([nome])
            if robo:
                robos[nome] = robo
                print(f"✅ {nome} conectado em {robo.address}")
        except Exception as e:
            print(f"❌ Falha ao conectar em {nome}: {e}")

# Executa a conexão
await conectar_equipe()

Buscando por: Fernanda...
Connected to CD:2C:91:2D:0D:0C
✅ Fernanda conectado em CD:2C:91:2D:0D:0C


In [21]:
if "Fernanda" in robos:
    # 1. Inicia o som
    await robos["Fernanda"].say("my2") 
    # 2. Roda a animação sem travar o código (create_task)
    task = asyncio.create_task(animar_olho(robos["Fernanda"], duracao=10.0))
    # 3. Espera a animação terminar antes de você ir para a próxima célula
    await task

## Finalize the study

In [4]:
async def disconnect_robots():
    for nome, r in robots.items():
        await r.eye_brightness(0) 
        await r.disconnect()
    print("All robots disconnected.")

await disconnect_robots()

NameError: name 'robots' is not defined